<a href="https://colab.research.google.com/github/kph4br/ds2002-fa26/blob/main/notebooks/01-foundations/2026-09-25-Cleaning-Gauntlet-Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
# TODO
before_rows = len(df)
n_dupes = df.duplicated().sum()
df = df.drop_duplicates()

log('dedup', f'dropped {n_dupes} exact duplicate rows', n_dupes)

# Notes for myself: the generator injected 15 duplicate rows on purpose, and
# 15 were found and dropped, leaving 300 rows (was 315).

[dedup] dropped 15 exact duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
# TODO
print(df['price'].value_counts())

df['price'] = df['price'].astype(str).str.replace('$', '', regex=False).astype(float)

log('clean price', 'stripped $ signs and cast price to float', len(df))

# Coerced all prices to be floats, and some had inconsistant use of
# the $. Now changed so every row has a numeric price from 6 to 24.

price
6.0       67
$12.00    65
$7.50     59
7.5       58
24        51
Name: count, dtype: int64
[clean price] stripped $ signs and cast price to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO
print(df['qty'].value_counts(dropna=False))

n_before = len(df)
bad_qty = df['qty'].isna() | (df['qty'] < 0)
n_bad = bad_qty.sum()
df = df.loc[~bad_qty].copy()
df['qty'] = df['qty'].astype(int)

log('clean qty', f'dropped {n_bad} rows with missing or negative qty', n_bad)

# 25 of the 300 rows had a missing or negative quantity so they were dropped.
# 275 rows remaining.

qty
 1.0    159
 2.0     69
 3.0     47
-1.0     13
 NaN     12
Name: count, dtype: int64
[clean qty] dropped 25 rows with missing or negative qty (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(df['item'].value_counts())

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger', 'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger', 'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho', 'rain poncho': 'Rain Poncho',
}

n_changed = (df['item'] != df['item'].map(ITEM_MAP)).sum()
df['item'] = df['item'].map(ITEM_MAP)

log('canon item', 'collapsed 6 spellings into 3 items', n_changed)

# Changed from 6 different ways of mixing capitalization to a standardized 3.

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[canon item] collapsed 6 spellings into 3 items (126 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
# TODO
print(df['category'].value_counts())

# I'm putting Apparel into "merch"
CATEGORY_MAP = {
    'Food': 'food', 'food': 'food',
    'Merch': 'merch', 'Apparel': 'merch',
    'RainGear': 'raingear', 'rain-gear': 'raingear',
}

n_changed_cat = (df['category'] != df['category'].map(CATEGORY_MAP)).sum()
df['category'] = df['category'].map(CATEGORY_MAP)

log('normalize category', 'standardized lowercase and collapsed spelling variants; put Apparel into merch', n_changed_cat)

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[normalize category] standardized lowercase and collapsed spelling variants; put Apparel into merch (231 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].isin(['Cheeseburger', 'Foam Finger', 'Rain Poncho']).all()
assert df['category'].isin(['food', 'merch', 'raingear']).all()
print('clean:', df.shape)

print(f"Interpretation: all five checks passed, so there are no duplicate rows, no bad qty values, "
      f"price is numeric, and item/category each only have their 3 intended values.")

clean: (275, 5)
Interpretation: all five checks passed, so there are no duplicate rows, no bad qty values, price is numeric, and item/category each only have their 3 intended values.


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
# TODO
df['revenue'] = df['qty'] * df['price']
rev_by_category = (
    df.groupby('category', as_index=False)['revenue']
      .sum()
      .sort_values('revenue', ascending=False)
      .round(2)
)

total_revenue = round(df['revenue'].sum(), 2)
display(rev_by_category)
print(f"Total revenue: ${total_revenue:,.2f}")

,category,revenue
0,food,1656.0
1,merch,1572.0
2,raingear,1512.0


Total revenue: $4,740.00


**What I would tell the vendor:** Stock more food... It's the top category at 1,656.00 dollars, ahead of merch (1,572.00 dollars) and raingear (1,512.00 dollars), so it brings in the most money per game and should stay stocked!

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,dedup,dropped 15 exact duplicate rows,15
1,clean price,stripped $ signs and cast price to float,300
2,clean qty,dropped 25 rows with missing or negative qty,25
3,canon item,collapsed 6 spellings into 3 items,126
4,normalize category,standardized lowercase and collapsed spelling ...,231


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a) The cleaning step that changed my revenue total the most was TODO 1 (deduplication step) because Revenue went from 4,852.50 dollars down to 4,594.50 dollars (dropping by 258.00 dollars).

b) One decision where someone else could have chosen differently than me is when I combined Apparel into merch in TODO 5. A different choice could've been to keep Apparel as its own category, since a t-shirt isn't really the same kind of product as a keychain or a foam finger. If I had kept them separate, the merch total of 1,572.00 dollars would have split into two smaller numbers, so merch wouldn't have been as close to food in revenue anymore. I decided to merge them anyway because apparel is sold and stocked the same way as the rest of merch at these games, so treating it separately probably wouldn't change what I'd actually tell a vendor to stock.